In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

data_path = "/Users/bonsitukebeto/Library/CloudStorage/OneDrive-SharedLibraries-NorthwesternUniversity/Arvind Krishna - Data/All Calls by Month"
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

i = 0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows=5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows=5))
    i += 1

all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))

### Read all data files
df_main = pd.DataFrame(columns=list(common_cols))
i = 0
for f in files:
    if f.suffix.lower() == ".csv":
        dfi = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        dfi = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, dfi], ignore_index=True)
    i += 1

### Datetime conversion
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

df_allcallsdata = df_main.copy()

# Step 2: inbound calls
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])

def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

df_allcallsdata["TempCallType"] = df_allcallsdata.apply(classify_call, axis=1)
earliest_calltype = df_allcallsdata.groupby("Correlation ID").first().reset_index()[["Correlation ID", "TempCallType"]]
df_allcallsdata = df_allcallsdata.drop(columns=["TempCallType"])
df_allcallsdata = df_allcallsdata.merge(
    earliest_calltype.rename(columns={"TempCallType": "Inbound/Outbound"}),
    on="Correlation ID", how="left"
)

df_allcallsdata_inbound = df_allcallsdata[df_allcallsdata["Inbound/Outbound"] == "Inbound"].copy()

# Time features
df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
df_allcallsdata_inbound["DayOfWeek"] = df_allcallsdata_inbound["Start time"].dt.weekday + 1
df_allcallsdata_inbound["Month"] = df_allcallsdata_inbound["Start time"].dt.month
df_allcallsdata_inbound["Quarter"] = df_allcallsdata_inbound["Start time"].dt.quarter
df_allcallsdata_inbound["Year"] = df_allcallsdata_inbound["Start time"].dt.year

# Step 3: number descriptions (SHORTENED + SEPARATE IMMIGRATION NUMBERS)
number_map = {
    "13123478300": "Internal VM",
    "13123411070": "Main number",
    "13124235938": "Community Legal",
    "13122296300": "Front Desk",
    "1180": "English Queue",
    "13125068646": "English Menu",
    "13124312299": "Farmworker/Migrant",
    "13122296079": "Nursing Home",
    "13122296344": "Bankruptcy",
    "13122296071": "Criminal Records",
    "13125068647": "Spanish Menu",
    "13123478340": "Veterans Rights",
    "13122296014": "Markham Eviction",
    "13123478309": "HIV Intake",
    "13122296072": "Juvenile Expungement",
    "13124235904": "Austin Intake",
    "13123478347": "Immigration - Local",
    "18882652188": "Immigration - Toll-Free",
    "13124235900": "CLASP",
    "13123478392": "Education Law",
    "13124235909": "Fair Housing",
    "13124312101": "OP Appeals",
    "13122296073": "Trafficking Survivors",
    "18004459025": "Migrant Legal",
    "18884018200": "Nursing Home Alt"
}

df_allcallsdata_inbound["Number Description"] = (
    df_allcallsdata_inbound["Called number"].astype(str).map(number_map).fillna("Unknown")
)

# Step 4: call flow / legs 1–4
df_allcallsdata_inbound = df_allcallsdata_inbound.sort_values(["Correlation ID", "Start time"])

call_sequences = (
    df_allcallsdata_inbound.groupby("Correlation ID")["Number Description"]
    .apply(lambda x: [v for v in x if pd.notna(v)])
    .reset_index(name="Call Flow")
)
call_sequences["num_legs"] = call_sequences["Call Flow"].apply(len)
call_sequences_clean = call_sequences[call_sequences["num_legs"] <= 4].copy()

max_legs = call_sequences_clean["Call Flow"].apply(len).max()
for i in range(max_legs):
    call_sequences_clean[f"Number Description_{i+1}"] = call_sequences_clean["Call Flow"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# STEP 5: Merge first-leg time fields and add Date column
first_leg_info = df_allcallsdata_inbound.groupby("Correlation ID").first().reset_index()
first_leg_info = first_leg_info[["Correlation ID", "Hour", "DayOfWeek", "Month", "Quarter", "Year", "Start time"]] 
first_leg_info['Date'] = first_leg_info['Start time'].dt.date

call_sequences_merged = call_sequences_clean.merge(first_leg_info, on="Correlation ID", how="left")

# STEP 6: Calculate metrics BY Number Description
# Filter to weekdays only
df_weekdays_unique = call_sequences_merged[call_sequences_merged['DayOfWeek'] <= 5].copy()

# Calculate total number of weekdays in dataset
num_weekdays = df_weekdays_unique['Date'].nunique()

# === HOURLY BY NUMBER DESCRIPTION ===
hourly_by_number = df_weekdays_unique.groupby(['Hour', 'Number Description_1']).agg(
    Total_Calls=('Correlation ID', 'nunique')
).reset_index()
hourly_by_number['Avg_Calls_Per_Weekday_Hourly'] = hourly_by_number['Total_Calls'] / num_weekdays

# === DAILY BY NUMBER DESCRIPTION ===
daily_by_number = df_weekdays_unique.groupby(['DayOfWeek', 'Number Description_1']).agg(
    Total_Calls=('Correlation ID', 'nunique'),
    Num_Days=('Date', 'nunique')
).reset_index()
daily_by_number['Avg_Calls_Per_Weekday_Daily'] = daily_by_number['Total_Calls'] / daily_by_number['Num_Days']

# === MONTHLY BY NUMBER DESCRIPTION ===
monthly_by_number = df_weekdays_unique.groupby(['Month', 'Year', 'Number Description_1']).agg(
    Total_Calls=('Correlation ID', 'nunique'),
    Weekdays_This_Month=('Date', 'nunique')
).reset_index()
monthly_by_number['Avg_Calls_Per_Weekday_Monthly'] = (
    monthly_by_number['Total_Calls'] / monthly_by_number['Weekdays_This_Month']
)

# Merge all metrics back to main dataset
call_sequences_merged = call_sequences_merged.merge(
    hourly_by_number[['Hour', 'Number Description_1', 'Avg_Calls_Per_Weekday_Hourly']],
    on=['Hour', 'Number Description_1'], how='left'
)

call_sequences_merged = call_sequences_merged.merge(
    daily_by_number[['DayOfWeek', 'Number Description_1', 'Avg_Calls_Per_Weekday_Daily']],
    on=['DayOfWeek', 'Number Description_1'], how='left'
)

call_sequences_merged = call_sequences_merged.merge(
    monthly_by_number[['Month', 'Year', 'Number Description_1', 'Avg_Calls_Per_Weekday_Monthly']],
    on=['Month', 'Year', 'Number Description_1'], how='left'
)

# STEP 7: Calculate overall rankings for filtering (Top 5, Next 5, etc.)
# Calculate total calls per intake number (for ranking)
intake_totals = call_sequences_merged.groupby('Number Description_1')['Correlation ID'].nunique().reset_index()
intake_totals.columns = ['Number Description_1', 'Total_Calls_Overall']
intake_totals = intake_totals.sort_values('Total_Calls_Overall', ascending=False).reset_index(drop=True)
intake_totals['Rank'] = intake_totals.index + 1

# Create rank groups 
def get_rank_group(rank, total_count):
    if rank <= 5:
        return "Top 5"
    else:
        start = ((rank - 1) // 5) * 5 + 1
        end = min(((rank - 1) // 5 + 1) * 5, total_count)
        return f"Rank {start}-{end}"

intake_totals['Rank_Group'] = intake_totals.apply(
    lambda row: get_rank_group(row['Rank'], len(intake_totals)), axis=1
)

# Merge rankings back to main dataset
call_sequences_merged = call_sequences_merged.merge(
    intake_totals[['Number Description_1', 'Rank', 'Rank_Group', 'Total_Calls_Overall']],
    on='Number Description_1', how='left'
)

# Export the final results
inbound_call_workflow = call_sequences_merged
inbound_call_workflow.to_csv("Nov5_Corrected_AllCallsData_Inbound_Call.csv", index=False)

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 66)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)
Columns missing from at least one dataframe: {'Redirecting party UUID', 'Auto Attendant Key Pressed', 'Device owner UUID', 'Recall Type', 'Column1', 'Queue Type', 'Public Called IP Address', 'Answered Elsewhere', 'Call Recording Result', 'Call Recording Trigger', 'Original reason2', 'User', 'PSTN vendor name2', 'Hold Duration', 'Original called party UUID', 'External caller ID number', 'Call Recording Platform Name', 'Public Calling IP Address'}
{'Department ID', 'Releasing party', 'Site UUID', 'Network call ID', 'Call transfer time', 'Related reason', 'Redirecting number', 'Local call ID', '